# LightGBM LambdaRank · Adri (rol B)

Objetivo: mejorar el LightGBM de referencia (CV ≈ 0.653 con las columnas originales).

**Reglas (EQUIPO.md):**
- Cada snapshot se predice **solo con sus propias filas**: toda feature nueva se calcula con `groupby("sample_id")` (o un grupo más fino dentro del snapshot).
- Validación: los 5 folds de `Data/folds.csv` (por controlador). Se decide por la CV, no por Kaggle.
- Una feature solo entra si **mejora la CV**.

**Cómo usar el notebook:** ejecuta de arriba abajo una vez. Después, para probar algo, cambia la sección 3 (features) o `PARAMS` y vuelve a ejecutar la celda de experimentos. Cada experimento se apunta en `experiments/lgb_runs.csv`.

## 1. Configuración

In [ ]:
import subprocess, sys, time
import numpy as np, pandas as pd, lightgbm as lgb
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "Data"
for p in ("oof", "preds", "submissions", "models", "experiments"):
    (ROOT / p).mkdir(exist_ok=True)

NAME = "lgb_adri"   # nombre de los archivos de salida
TYPES = ['entry_conflict_resolution', 'exit_conflict_resolution', 'entry_coordination', 'exit_coordination',
         'assume_aircraft', 'transfer_aircraft', 'conflict_resolution', 'non_conformance_resolution',
         'qos_improvement', 'entry_conditions', 'exit_conditions', 'aircraft_request', 'zone_conflict',
         'return_to_route']

# Parámetros base de EQUIPO.md
PARAMS = dict(objective="lambdarank", n_estimators=400, learning_rate=0.05, num_leaves=31, min_child_samples=50,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.7, lambdarank_truncation_level=5,
              random_state=0, verbose=-1)

## 2. Datos
- `lab` = train + validation (con etiqueta), cada fila con su `fold` (0–4).
- `test` = test, sin etiqueta.
- `cs` = callsign del avión de la tarea (sacado de `task_name`). **No es una feature**: solo sirve para agrupar las tareas de un mismo avión.

In [ ]:
def load(name):
    d = pd.read_csv(DATA / f"{name}.csv")
    pref = d.task_index.map(lambda t: len(TYPES[t]) + 1)
    d["cs"] = [n[p:].split("_")[0] for n, p in zip(d.task_name, pref)]
    return d

lab = pd.concat([load("train"), load("validation")], ignore_index=True)
lab = lab.merge(pd.read_csv(DATA / "folds.csv"), on="atco", how="left")
test = load("test")
lab = lab.sort_values(["sample_id", "row_index"]).reset_index(drop=True)
test = test.sort_values(["sample_id", "row_index"]).reset_index(drop=True)

assert lab.fold.notna().all(), "hay controladores sin fold"
assert (lab.groupby("sample_id").target_index.sum() == 1).all(), "alguna muestra no tiene exactamente 1 respuesta"
print(f"lab: {lab.sample_id.nunique():,} muestras, {len(lab):,} filas | test: {test.sample_id.nunique():,} muestras")
print(lab.drop_duplicates("sample_id").groupby("fold").atco.unique())

## 3. Features
Cada grupo es una función que recibe el dataframe y devuelve **solo las columnas nuevas**. Todas agrupan por `sample_id`, así que solo usan el propio snapshot.

| Grupo | Qué añade |
|---|---|
| `snapshot` | Ranking y diferencia respecto al mínimo **dentro del snapshot** de varias columnas de urgencia/actividad |
| `tipo` | Los mismos rankings **dentro de las tareas del mismo tipo** del snapshot |
| `avion` | Si la tarea es la más antigua de su avión, cuántos tipos distintos tiene el avión |
| `tipo_estado` | Cruces tipo × estado: assume con oferta pendiente, transfer debido, zona con su distancia, etc. |
| `contexto` | Cómo está el snapshot: nº de aviones, nº de transfers/assumes/zonas/conflictos pendientes |

Para añadir una idea nueva: escribe otra función `fe_...` y añádela a `FEATURE_GROUPS`.

In [ ]:
META = ['sample_id', 'target_index', 'session', 'atco', 'scenario', 'timestamp', 'task_name', 'row_index',
        'fold', 'cs']
BASE = [c for c in lab.columns if c not in META]
RANK_COLS = ["a_since_flight", "x_task_age", "x_state_age", "time_to_MD", "k_timeToExitPoint",
             "distance_to_zone", "abs_tts"]


def _with_abs(df):
    return df.assign(abs_tts=df.time_to_sector.abs())


def fe_snapshot(df):
    x = _with_abs(df)
    g = x.groupby("sample_id")
    out = {}
    for c in RANK_COLS:
        out[f"rs_{c}"] = g[c].rank(method="min")
        out[f"ds_{c}"] = x[c] - g[c].transform("min")
    return pd.DataFrame(out, index=df.index)


def fe_tipo(df):
    x = _with_abs(df)
    g = x.groupby(["sample_id", "task_index"])
    out = {}
    for c in RANK_COLS:
        out[f"rt_{c}"] = g[c].rank(method="min")
        out[f"dt_{c}"] = x[c] - g[c].transform("min")
    return pd.DataFrame(out, index=df.index)


def fe_avion(df):
    g = df.groupby(["sample_id", "cs"])
    out = {
        "ac_age_rank": g.x_task_age.rank(method="min", ascending=False),   # 1 = la tarea más antigua del avión
        "ac_n_types": g.task_index.transform("nunique"),
        "ac_has_assume": g.task_index.transform(lambda s: (s == 4).any()).astype(int),
    }
    return pd.DataFrame(out, index=df.index)


def fe_tipo_estado(df):
    t = df.task_index
    out = {
        "ts_assume_offered": ((t == 4) & (df.x_transfer_in == 1)).astype(int),
        "ts_transfer_due": ((t == 5) & (df.k_aircraftNeedsToBeTransferred == 1)).astype(int),
        "ts_entry_coord_done": ((t == 2) & (df.x_isc_coordinated == 1)).astype(int),
        "ts_entry_notified": ((t == 2) & (df.x_isc_notified == 1)).astype(int),
        "ts_zone_dist": df.distance_to_zone.where(t == 12),
        "ts_conflict_ttmd": df.time_to_MD.where(t.isin([0, 1, 6])),
        "ts_twin_assume": ((t == 4) & (df.l_twin_AE == 1)).astype(int),
        "ts_twin_entry": ((t == 2) & (df.l_twin_AE == 1)).astype(int),
    }
    return pd.DataFrame(out, index=df.index)


def fe_contexto(df):
    g = df.groupby("sample_id")
    tmp = pd.DataFrame({
        "sample_id": df.sample_id,
        "is_assume_offer": ((df.task_index == 4) & (df.x_transfer_in == 1)).astype(int),
        "is_transfer_due": ((df.task_index == 5) & (df.k_aircraftNeedsToBeTransferred == 1)).astype(int),
        "is_zone": (df.task_index == 12).astype(int),
        "is_conflict": df.task_index.isin([0, 1, 6]).astype(int),
        "is_entry_coord": (df.task_index == 2).astype(int),
    })
    gt = tmp.groupby("sample_id")
    out = {"cx_n_aircraft": g.cs.transform("nunique")}
    for c in tmp.columns[1:]:
        out["cx_n_" + c[3:]] = gt[c].transform("sum")
    return pd.DataFrame(out, index=df.index)


FEATURE_GROUPS = {"snapshot": fe_snapshot, "tipo": fe_tipo, "avion": fe_avion,
                  "tipo_estado": fe_tipo_estado, "contexto": fe_contexto}


def build(df, groups):
    """Devuelve (df con las columnas nuevas, lista de features a usar)."""
    new = [FEATURE_GROUPS[g](df) for g in groups]
    out = pd.concat([df] + new, axis=1)
    return out, BASE + [c for n in new for c in n.columns]

## 4. Entrenamiento y CV
- `fit`: entrena un `LGBMRanker`. Le dice a LightGBM qué filas forman cada snapshot (`group`) para que aprenda a **ordenar dentro del snapshot**.
- `hit3`: para cada snapshot, 1 si la respuesta está entre las 3 de mayor score.
- `cv`: para cada fold, entrena con los otros 4 y predice ese. El resultado es el **OOF** (out-of-fold): una predicción para cada fila hecha por un modelo que no la vio.

In [ ]:
def fit(A, F, params):
    m = lgb.LGBMRanker(**params)
    m.fit(A[F], A.target_index, group=A.groupby("sample_id", sort=False).size().values,
          categorical_feature=["task_index"])
    return m


def hit3(d, score):
    t = d[["sample_id", "target_index"]].assign(s=np.asarray(score))
    t = t.sort_values(["sample_id", "s"], ascending=[True, False])
    return t.groupby("sample_id").head(3).groupby("sample_id").target_index.max()


def cv(d, F, params):
    oof = np.full(len(d), np.nan)
    for f in range(5):
        tr, va = (d.fold != f).values, (d.fold == f).values
        oof[va] = fit(d[tr], F, params).predict(d.loc[va, F])
    h = hit3(d, oof)
    fold_of = d.drop_duplicates("sample_id").set_index("sample_id").fold
    return oof, h, h.groupby(fold_of.reindex(h.index)).mean()


LOG = ROOT / "experiments" / "lgb_runs.csv"


def experimento(nombre, groups, params=None, nota=""):
    params = {**PARAMS, **(params or {})}
    t0 = time.time()
    d, F = build(lab, groups)
    oof, h, por_fold = cv(d, F, params)
    fila = dict(hora=time.strftime("%H:%M"), nombre=nombre, grupos="+".join(groups) or "-",
                params_cambiados=str({k: v for k, v in params.items() if PARAMS.get(k) != v}),
                n_features=len(F), cv_top3=round(h.mean(), 4), std_folds=round(por_fold.std(), 4), nota=nota)
    pd.DataFrame([fila]).to_csv(LOG, mode="a", header=not LOG.exists(), index=False)
    print(f"{nombre:28s} CV top-3 = {h.mean():.4f}  (folds: {' '.join(f'{v:.3f}' for v in por_fold)})"
          f"  {len(F)} features  {time.time() - t0:.0f}s")
    return oof, h

## 5. Experimentos
Primero la referencia (columnas originales). Después cada grupo de features **por separado**, para ver cuál aporta. Por último, todos juntos.

Diferencias menores de ~0.003 pueden ser ruido: fíjate también en si mejora en la mayoría de folds.

In [ ]:
oof_base, h_base = experimento("base", [])
for g in FEATURE_GROUPS:
    experimento(f"base+{g}", [g])
oof_all, h_all = experimento("base+todos", list(FEATURE_GROUPS))

In [ ]:
# Historial de todos los experimentos (el más reciente al final)
pd.read_csv(LOG).tail(15)

## 6. Dónde falla
Acierto según **el tipo de la tarea correcta**. Útil para decidir qué features atacar (y para Pau).

In [ ]:
def por_tipo(h):
    y = lab[lab.target_index == 1].set_index("sample_id").task_index.map(lambda t: TYPES[t])
    return (pd.DataFrame({"tipo": y.reindex(h.index), "hit": h})
              .groupby("tipo").hit.agg(acierto="mean", muestras="size")
              .sort_values("muestras", ascending=False).round(3))

pd.concat({"base": por_tipo(h_base), "todos": por_tipo(h_all)}, axis=1)

## 7. Modelo final: guardar OOF, modelo y predicciones
Cuando tengas la mejor combinación, pon aquí `BEST_GROUPS` y `BEST_PARAMS` y ejecuta. Genera:
- `oof/lgb_adri.csv` — predicciones OOF (para el stacking de Juanma y el análisis de Pau)
- `models/lgb_adri.txt` — **los pesos**: el modelo entrenado con todo train+validation
- `preds/lgb_adri.csv` y `submissions/sub_lgb_adri.csv` — predicciones de test

In [ ]:
BEST_GROUPS = list(FEATURE_GROUPS)   # <- cambiar por la mejor combinación
BEST_PARAMS = {}                     # <- parámetros cambiados respecto a PARAMS

params = {**PARAMS, **BEST_PARAMS}
d, F = build(lab, BEST_GROUPS)
oof, h, _ = cv(d, F, params)
print(f"CV top-3 final: {h.mean():.4f}")
d[["sample_id", "row_index"]].assign(score=oof).to_csv(ROOT / "oof" / f"{NAME}.csv", index=False)

m = fit(d, F, params)
m.booster_.save_model(str(ROOT / "models" / f"{NAME}.txt"))

dt, _ = build(test, BEST_GROUPS)
pred = dt[["sample_id", "row_index"]].assign(score=m.predict(dt[F]))
pred.to_csv(ROOT / "preds" / f"{NAME}.csv", index=False)
sub = (pred.sort_values(["sample_id", "score"], ascending=[True, False]).groupby("sample_id").row_index
           .apply(lambda r: " ".join(map(str, r.iloc[:3]))).rename("prediction").reset_index())
sub.to_csv(ROOT / "submissions" / f"sub_{NAME}.csv", index=False)
print("Guardado:", NAME, "| test:", len(sub), "muestras")

### Comprobación con el scorer oficial
Con el OOF construimos un envío solo para las muestras de `validation.csv` y lo puntúa `Data/scorer.py`. Debe dar lo mismo que nuestro `hit3` en esas muestras.

In [ ]:
val_ids = set(pd.read_csv(DATA / "validation.csv", usecols=["sample_id"]).sample_id)
o = d[["sample_id", "row_index"]].assign(score=oof)
o = o[o.sample_id.isin(val_ids)].sort_values(["sample_id", "score"], ascending=[True, False])
o.groupby("sample_id").row_index.apply(lambda r: " ".join(map(str, r.iloc[:3]))).rename("prediction") \
 .reset_index().to_csv(ROOT / "submissions" / f"val_{NAME}.csv", index=False)
print(f"hit3 propio en validation: {h[h.index.isin(val_ids)].mean():.4f}")
r = subprocess.run([sys.executable, DATA / "scorer.py", "--labelled", DATA / "validation.csv",
                    ROOT / "submissions" / f"val_{NAME}.csv"], capture_output=True, text=True)
print("scorer oficial:", r.stdout or r.stderr)

### Cómo usan los demás el modelo guardado (sin entrenar)
```python
m = lgb.Booster(model_file="models/lgb_adri.txt")
dt, F = build(test, BEST_GROUPS)        # mismas features que en el entrenamiento
score = m.predict(dt[F])
```